# Lab 3.3 — MapReduce and Spark on a mini-cluster of your own
**Data Engineering · Block 3 · Distributed Systems**

**Student:** `<your name>` &nbsp;&nbsp; **Date:** `<date>`

> **Policy** (same as labs 1.3 and 1.4): no AI tools, no plagiarism. The instructor may ask for an individual walkthrough of any part.

## What you will do
You will run a **real Spark standalone cluster** on your Codespace: 1 *master* + 2 *workers* (separate JVM processes, each with its own "disk" and IP address) and **this notebook as the driver**. On it you will first re-invent MapReduce by hand, then use Spark and *measure* what the lectures of Block 3 told you: partitioning, shuffles, hot keys, joins, fault tolerance and the cost of distribution.

| Part | Topic | Lectures |
|---|---|---|
| 0 | Your cluster | 3.1 |
| 1 | MapReduce by hand (map, partition, shuffle, combiner) | 3.3 |
| 2 | From MapReduce to Spark RDDs | 3.3 |
| 3 | DataFrames, query plans and the Spark UI | 3.3 |
| 4 | Joins and data skew | 3.2, 3.3 |
| 5 | Failures and the cost of distribution | 3.3, 3.4 (preview), 3.7 (preview) |

## How to work
- 🔧 cells: complete the code (replace each `...`). ✍️ cells: write your answer *in your own words* (2–5 lines unless stated).
- Do **not** delete or reorder the provided cells. **Run the notebook top to bottom before submitting.**
- Timings: **the first Spark job after a restart is always slower** (JIT compilation, starting Python workers). Always run a query twice and report the *second* run.
- Your numbers will differ from your classmates' (different machines, noisy neighbours). What must agree is *the reasoning*.
- ★ marks a **bonus** item (not required for full marks).

In [ ]:
import re, time, zlib, math
from collections import Counter, defaultdict
from operator import add
from pathlib import Path

import duckdb
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

from lab33_utils import *      # cluster helpers: start_cluster(), get_spark(), stages_df(), ... (read the file when you have time)

check_environment()            # what does this machine offer? (Java, Python, cores, RAM)

---
# Part 0 — Your cluster

Recall 3.1: a cluster has a **control plane** (decides who does what) and a **data plane** (does the work). Spark's *standalone* mode has exactly these roles:

- **Master** — keeps the list of workers, accepts applications and *grants resources* (it does not touch your data).
- **Worker** — one per machine; launches **executors** for the applications that the master assigns to it.
- **Executor** — a JVM process that runs **tasks** on partitions of the data and keeps shuffle files on its local disk.
- **Driver** — your program (this notebook): builds the plan, splits it into stages and tasks, and talks **directly** to the executors.

`start_cluster()` launches the master and two workers as separate OS processes. Each worker gets its own directory (its "disk") and its own loopback address (`127.0.0.2`, `127.0.0.3`), so Spark believes they are different hosts — as in a real cluster.

In [ ]:
t0 = time.perf_counter()
start_cluster(workers=2, cores=1, memory="1g")
cluster_start_s = time.perf_counter() - t0       # we will need this number in Part 5
cluster_status()

In [ ]:
t0 = time.perf_counter()
spark = get_spark("lab33")                       # connects this notebook (the driver) to the master
session_start_s = time.perf_counter() - t0       # ... and this one
sc = spark.sparkContext
ps_spark()                                       # the OS processes behind your cluster

In [ ]:
paths = make_dataset("data", n_rows=5_000_000, files=8)    # synthetic ride-hailing trips (Parquet), ~3 s
print(paths)

### 🔧 Open the two web UIs
In VS Code open the **PORTS** tab (next to *Terminal*). Open port **8080** (*Spark Master UI*) and port **4040** (*Spark application UI*) in the browser.
*(Links inside the Master UI that point to `127.0.0.1` do not work through Codespaces' port forwarding — ignore them and use the PORTS tab.)*

📸 **Screenshot 1** — the Master UI showing **2 ALIVE workers** and your application under *Running Applications*. Save it as `evidence/1_master_ui.png` and run the next cell.

In [ ]:
show_evidence("evidence/1_master_ui.png")

**✍️ Q0** — Complete the table with what you saw in `ps_spark()` and in the Master UI (3.1 vocabulary). In the last column answer: *if this process died while a job is running, does the job survive?* (You will test the answers in Part 5.)

| Component | OS process(es) (role in `ps_spark()`) | Plane (control / data) | Job survives its death? |
|---|---|---|---|
| Master | | | |
| Worker | | | |
| Executor | | | |
| Driver | | | |

> *Write your answer here (double-click to edit, Shift+Enter to render).*

---
# Part 1 — MapReduce by hand

No Spark yet. The function `mapreduce()` below is a **single-process simulation** of the execution flow in Dean & Ghemawat (2004, §3.1):

1. the input is cut into **M splits** (contiguous chunks, like HDFS blocks);
2. each **map task** reads one split and writes its output into **R buckets** (its "intermediate files"), choosing the bucket with `hash(key) mod R`;
3. an optional **combiner** pre-aggregates each map task's output *before* it leaves the machine;
4. the **shuffle**: reducer *r* fetches bucket *r* from **every** map task (this is the only step that uses the network);
5. each **reduce task** merges what it fetched, sorts by key and applies the reduce function.

The data: ~10,000 lines of English text (the Python language reference).

In [ ]:
lines = corpus_lines()
print(f"{len(lines):,} lines. Example: {lines[100]!r}")

def partition(key: str, R: int) -> int:
    """Which reducer receives this key?  hash(key) mod R, as in the paper.
    We use crc32 and NOT Python's hash(): hash() of a str is randomised per process,
    so two map processes would disagree about where a key goes!"""
    return zlib.crc32(key.encode()) % R


def mapreduce(records, map_fn, reduce_fn, M=4, R=4, combine_fn=None):
    """Single-process simulation of a MapReduce job. Returns (output, stats)."""
    # 1. Input splits: M contiguous chunks
    size = math.ceil(len(records) / M)
    splits = [records[i * size:(i + 1) * size] for i in range(M)]

    # 2. Map phase: map task m reads split m and writes R buckets
    buckets = [[defaultdict(list) for _ in range(R)] for _ in range(M)]
    emitted = 0
    for m, split in enumerate(splits):
        for record in split:
            for key, value in map_fn(record):
                buckets[m][partition(key, R)][key].append(value)
                emitted += 1
        # 3. Combiner (optional): runs on the map side, once per bucket
        if combine_fn:
            for r in range(R):
                buckets[m][r] = {k: [combine_fn(k, vs)] for k, vs in buckets[m][r].items()}

    # 4. Shuffle: how many (key, value) pairs must travel to the reducers?
    shuffled = sum(len(vs) for m in range(M) for r in range(R) for vs in buckets[m][r].values())

    # 5. Reduce phase: reducer r merges bucket r of every map task, sorts by key, reduces
    output, load = {}, []
    for r in range(R):
        merged = defaultdict(list)
        for m in range(M):
            for key, values in buckets[m][r].items():
                merged[key].extend(values)
        load.append(sum(len(v) for v in merged.values()))
        for key in sorted(merged):
            k, v = reduce_fn(key, merged[key])
            output[k] = v
    return output, {"emitted": emitted, "shuffled": shuffled, "reducer_load": load}

**✍️ Q1.0** — Read `mapreduce()`. (a) Which of its 5 steps would run **in parallel** on a real cluster, and how many parallel tasks could there be in each? (b) Which step is the only one that needs the **network**? (c) Name **one thing a real framework does that this simulation does not** (look at the lecture on fault tolerance).

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 1a — 🔧 Word count
Write the map and reduce functions. Contract: `map_fn(record) -> list of (k2, v2)`, `reduce_fn(k2, list_of_v2) -> (k2, v3)`.

In [ ]:
def wc_map(line):
    """line -> [(word, 1), ...]"""
    return ...          # hint: re.findall(r"[a-z]+", line.lower())

def wc_reduce(word, counts):
    """(word, [1, 1, 1, ...]) -> (word, total)"""
    return ...

counts, stats = mapreduce(lines, wc_map, wc_reduce, M=4, R=4)
truth = Counter(w for l in lines for w in re.findall(r"[a-z]+", l.lower()))
assert counts == dict(truth), "your MapReduce result differs from collections.Counter"
print(stats)
pd.Series(counts).sort_values(ascending=False).head(5)

### 1b — 🔧 The combiner
A combiner has the **same signature as the reducer's input and output types**: it receives `(k2, list of v2)` and returns **one v2**. Write one for word count and compare the traffic.

In [ ]:
def wc_combine(word, counts):
    return ...

counts_c, stats_c = mapreduce(lines, wc_map, wc_reduce, M=4, R=4, combine_fn=wc_combine)
assert counts_c == counts

summary = pd.DataFrame({"pairs emitted": [stats["emitted"], stats_c["emitted"]],
                        "pairs shuffled": [stats["shuffled"], stats_c["shuffled"]]},
                       index=["no combiner", "with combiner"])
print(summary)
print(f"shuffle reduced by a factor of {stats['shuffled'] / stats_c['shuffled']:.1f}")

# Why can't it shrink further?  Lower bound = number of distinct (map task, word) pairs
size = math.ceil(len(lines) / 4)
splits = [lines[i * size:(i + 1) * size] for i in range(4)]
lower_bound = ...            # one line: sum over the 4 splits of the number of DISTINCT words in the split
print("lower bound:", lower_bound, "== shuffled with combiner?", lower_bound == stats_c["shuffled"])

**✍️ Q1b** — (i) By what factor did the shuffle shrink? (ii) Why is `lower_bound` the minimum possible with a combiner? (iii) Why did the final result **not** change? (iv) Which two properties of the word-count operation make this safe?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 1c — 🔧 The combiner trap: the average
Task: the **average word length per initial letter** (`'a'`, `'b'`, ...). The straightforward solution maps every word to `(initial, length)` and lets the reducer average. We want to use the same trick as in 1b, a combiner that averages. **Is it correct?**

In [ ]:
def len_map(line):
    return [(w[0], len(w)) for w in re.findall(r"[a-z]+", line.lower())]

def mean_reduce(letter, values):
    return (letter, sum(values) / len(values))

good, _ = mapreduce(lines, len_map, mean_reduce, M=4, R=4)                      # no combiner: correct by construction

def mean_combine(letter, values):
    return sum(values) / len(values)                                            # the "obvious" combiner

bad, _ = mapreduce(lines, len_map, mean_reduce, M=4, R=4, combine_fn=mean_combine)
err = pd.Series({k: abs(good[k] - bad[k]) for k in good})
print(f"letters with a wrong result: {(err > 1e-9).sum()} of {len(err)};  largest error: {err.max():.3f} characters (letter {err.idxmax()!r})")

# --- the fix: carry (sum, count) instead of the mean, and divide only in the reducer
def len_map2(line):
    return ...                  # (initial, (length, 1))

def sumcount_combine(letter, values):
    return ...                  # (sum of lengths, sum of counts)

def sumcount_reduce(letter, values):
    return ...                  # (letter, total / n)

fixed, stats_fix = mapreduce(lines, len_map2, sumcount_reduce, M=4, R=4, combine_fn=sumcount_combine)
assert all(abs(fixed[k] - good[k]) < 1e-9 for k in good)
print("fixed == good:", True, "| pairs shuffled:", stats_fix["shuffled"])

**✍️ Q1c** — (i) Why is the average of the averages wrong? Give the one-line reason in terms of *how many words each map task saw*. (ii) State the property that a combine function must satisfy. (iii) Is `max` safe to combine? Is the **median**? (iv) In Part 3 you will see `partial_avg` in a Spark plan: how does Spark ship an average through a shuffle?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 1d — 🔧 Load on the reducers
`stats["reducer_load"]` is the number of pairs each of the R = 4 reducers had to process.

In [ ]:
def imbalance(load):
    """max load / mean load (1.0 = perfectly balanced)"""
    return ...

print("no combiner :", stats["reducer_load"], f"-> imbalance {imbalance(stats['reducer_load']):.2f}")
print("with combiner:", stats_c["reducer_load"], f"-> imbalance {imbalance(stats_c['reducer_load']):.2f}")

heaviest = max(range(4), key=lambda r: stats["reducer_load"][r])
print("heaviest reducer:", heaviest, "| most frequent words:", truth.most_common(3))
print("reducer of 'the':", partition("the", 4))

**✍️ Q1d** — Which word makes one reducer heavier? Connect this to the *hot key* problem of lecture 3.2 ("hashing evens out keys, not popularity"). Why did the combiner help here, and what would it **not** fix?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

---
# Part 2 — From MapReduce to Spark RDDs

Spark's first abstraction, the **RDD** (Zaharia et al., 2012), is MapReduce with a friendlier syntax: `flatMap`/`map` play the role of the map phase and `reduceByKey` is *shuffle + combine + reduce* in one call. Now the work really runs on your 2 executors.

In [ ]:
corpus_path = write_corpus("data/corpus.txt", copies=40)      # the same text, 40 times: 22 MB, ~3 million words
text = sc.textFile(corpus_path, minPartitions=8)
print("partitions of the input:", text.getNumPartitions())

### 2a — 🔧 Word count in Spark
Same logic as Part 1a. Nothing runs until the **action** `collectAsMap()`.

In [ ]:
def word_count():
    """Builds a NEW RDD each time it is called (we will see why in 2b-bis)."""
    return (text.flatMap(...)                # line -> list of words
                .map(...)                    # word -> (word, 1)
                .reduceByKey(...))           # hint: operator.add is already imported as `add`

for attempt in (1, 2):                       # run twice: the 2nd run is the "warm" one
    s0 = next_stage_id(spark)                # remember where we are in the list of stages
    with timed(f"Spark word count, run {attempt}"):
        result = word_count().collectAsMap()
assert len(result) == len(truth) and all(result[w] == 40 * c for w, c in truth.items())
new_stages(spark, s0)                        # the stages of the LAST run

**✍️ Q2a** — Compare the stages above with **M** and **R** of Part 1. How many tasks does each stage have, and who decided that number? How much data crossed the shuffle (`shuffle_write_MB` of the first stage)? Which structure of Part 1 does that correspond to?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 2b — Lineage and stages
An RDD remembers **how it was computed from its parents** (its *lineage*). Spark cuts the lineage into **stages** at every shuffle.

In [ ]:
wc = word_count()
print(wc.toDebugString().decode())

**✍️ Q2b** — (i) Where is the stage boundary in the lineage above? (ii) Which operations are *pipelined* inside the first stage (executed one record at a time, without materialising intermediate collections)? (iii) Nothing ran when you wrote `words = ...`, `pairs = ...`, `wc = ...`. What is that property called and what does it buy Spark? (iv) If an executor dies and loses a partition, how can Spark rebuild it without replicas?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

**★ 2b-bis — Why did `word_count()` build a new RDD each time?** Run an action twice on the **same** RDD object `wc` and look at *all* the stages, including the skipped ones.

In [ ]:
s0 = next_stage_id(spark)
with timed("1st action on wc"):
    wc.collectAsMap()
with timed("2nd action on the SAME wc"):
    wc.collectAsMap()
stages_df(spark, include_skipped=...).query("stage >= @s0")      # show the SKIPPED stages too

**✍️ ★ Q2b-bis** — Which stage is `SKIPPED` and why can Spark skip it? What does this teach you about benchmarking Spark code, and about fault tolerance (Part 5)?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 2c — 🔧 The combiner at scale: `reduceByKey` vs `groupByKey`
We need the **total fare per zone** on a 20 % sample of the trips (1 million rows). Two ways to write it:
`reduceByKey(add)` (combines on the map side) and `groupByKey().mapValues(sum)` (ships **every** value to the reducer and sums there).

In [ ]:
trips = spark.read.parquet(paths["trips"])
fares = (trips.select("zone_id", "fare_eur").sample(fraction=0.2, seed=7)
              .rdd.map(lambda r: (r.zone_id, r.fare_eur)))

def bench_rdd(label, action, reps=2):
    s0 = next_stage_id(spark)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter(); action(); times.append(time.perf_counter() - t0)
    st = new_stages(spark, s0)
    shuffle_mb = st[st.shuffle_write_MB > 0].shuffle_write_MB.iloc[-1]     # shuffle volume of the last run
    return {"variant": label, "seconds (2nd run)": round(times[-1], 2), "shuffle_write_MB": shuffle_mb}

rows = [bench_rdd("reduceByKey(add)", lambda: ...),                         # sum of fares per zone, then .count() as the action
        bench_rdd("groupByKey().mapValues(sum)", lambda: ...)]
rdd_table = pd.DataFrame(rows)
rdd_table

**✍️ Q2c** — Explain the difference in `shuffle_write_MB` using Part 1b (by what factor?). Which variant would you use for a **sum**, and which one is unavoidable for a **median per key**? (Why?)

> *Write your answer here (double-click to edit, Shift+Enter to render).*

---
# Part 3 — DataFrames, query plans and the Spark UI

Most real Spark code uses **DataFrames**: tables with a schema. Because Spark knows the structure, its optimiser (**Catalyst**, Armbrust et al., 2015) can rewrite your query and generate efficient JVM code (**Tungsten**) — and you can *read the plan it chose*.

In [ ]:
trips = spark.read.parquet(paths["trips"])
trips.printSchema()
print("input partitions:", trips.rdd.getNumPartitions())
disk_mb = sum(f.stat().st_size for f in Path(paths["trips"]).glob("*.parquet")) / 1e6
print(f"{trips.count():,} rows, {disk_mb:.0f} MB on disk, cores in the cluster: {sc.defaultParallelism}")

### 3a — 🔧 How many map tasks? Input partitions
Spark decides the number of input partitions from the size of the files, `spark.sql.files.maxPartitionBytes` (default 128 MiB) and the number of cores. Change `maxPartitionBytes` to **8 MiB**, re-read the data and compare how long the same aggregation takes.

In [ ]:
def total_fare_by_zone(df):
    return df.groupBy("zone_id").agg(F.sum("fare_eur").alias("fare"))

def best_of(df, reps=3):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); run(df); ts.append(time.perf_counter() - t0)
    return round(min(ts), 2)

rows = []
for mb in (128, 8):
    spark.conf.set(..., str(mb * 1024 * 1024))           # which property?
    t = spark.read.parquet(paths["trips"])
    rows.append({"maxPartitionBytes (MiB)": mb, "input partitions": ...,
                 "best of 3 (s)": best_of(total_fare_by_zone(t))})
pd.DataFrame(rows)

**✍️ Q3a** — (i) With the default setting, why are there so few input partitions although there are 8 files? (ii) Which setting gave the faster query and why — would the answer change on a 100-core cluster? (iii) The MapReduce paper recommends 16–64 MB per split: what happens if splits are too small? Too large?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 3b — 🔧 Reading a plan
Run the query below and read its **physical plan**. Then map each node to the MapReduce vocabulary of Part 1.

In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))     # back to the default
trips = spark.read.parquet(paths["trips"])

q = (trips.filter(F.col("distance_km") > 1)
          .groupBy("zone_id")
          .agg(F.count("*").alias("n"), F.avg("fare_eur").alias("avg_fare")))
q.explain("formatted")

**✍️ Q3b** — Complete the mapping. For each plan node, say which MapReduce element of Part 1 it corresponds to (**map**, **combiner**, **shuffle**, **reduce**) and what it does here. Also answer: (i) which filter was *pushed down* to the Parquet reader and which columns does the scan read (look for `PushedFilters` and `ReadSchema`)? (ii) Why does the plan compute `partial_count(1)`, and a `sum` and a `count` for `avg`? (relate to 1c)

| Plan node | MapReduce element |
|---|---|
| `Scan parquet` + `Filter` + `Project` | |
| `HashAggregate` (with `partial_` functions) | |
| `Exchange hashpartitioning(zone_id, ...)` | |
| `HashAggregate` (final) | |

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 3c — 🔧 The Spark UI
Run the query **once** and explore it in the **application UI** (port 4040). Start with the **SQL / DataFrame** tab: click your query (the most recent one) and open *Plan Visualization*: the plan as a graph, with the number of rows at every node. Then look at the **Jobs** and **Stages** tabs.

📸 **Screenshot 2** — the plan graph of this query (SQL / DataFrame tab). Save it as `evidence/2_spark_ui_plan.png`.

*(The Jobs tab may list a stage as **skipped**: with adaptive execution a query can run as several jobs, and a stage already computed by a previous job is not run again — the same shuffle-file reuse you saw in 2b-bis.)*

In [ ]:
s0 = next_stage_id(spark)
run(q)
st = ...                 # the stages created by this execution (hint: new_stages)
st

In [ ]:
show_evidence("evidence/2_spark_ui_plan.png")

**✍️ Q3c** — (i) How many **stages** did the execution create, and which one writes shuffle data? (ii) How many **tasks** has each stage, and why does the last one have so few? (hint: the plan said `200` partitions and `isFinalPlan=false`; look at `AQEShuffleRead` in the plan graph) (iii) In the plan graph: how many rows does the scan produce, how many reach the `Exchange` (*shuffle records written*), and which idea from Part 1 explains the difference?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 3d — 🔧 Shuffle partitions and Adaptive Query Execution
Spark creates `spark.sql.shuffle.partitions` (default **200**) reduce tasks after a shuffle, whatever the data size. Since 3.2, AQE (`spark.sql.adaptive.enabled`, on by default) can merge small partitions at run time. Run the same query under three configurations.

In [ ]:
def timed_query(label, df, reps=2):
    s0 = next_stage_id(spark)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); run(df); ts.append(time.perf_counter() - t0)
    st = new_stages(spark, s0)
    reduce_tasks = int(st[st.shuffle_read_MB > 0].tasks.iloc[-1])
    return {"configuration": label, "reduce tasks": reduce_tasks, "seconds (2nd run)": round(ts[-1], 2)}

def agg_query():
    return (spark.read.parquet(paths["trips"]).filter(F.col("distance_km") > 1)
                 .groupBy("zone_id").agg(F.count("*").alias("n"), F.avg("fare_eur").alias("avg_fare")))

rows = []
# Configuration 1: AQE off, 200 shuffle partitions
...
# Configuration 2: AQE off, 4 shuffle partitions
...
# Configuration 3: AQE on, 200 shuffle partitions (the default)
...
pd.DataFrame(rows)

**✍️ Q3d** — Explain the three rows. Why is 200 a poor value for this query? What did AQE do, and why do you *still* need to know about `spark.sql.shuffle.partitions` on a large production cluster?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 3e — The same aggregation with RDDs and with DataFrames
In 2c you computed *total fare per zone* with RDDs on a **20 % sample (1 M rows)**. Compare it with the DataFrame version on **all 5 M rows**.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
df_time = best_of(total_fare_by_zone(spark.read.parquet(paths["trips"])))
rdd_time = rdd_table.loc[0, "seconds (2nd run)"]
print(f"RDD reduceByKey on 1M rows : {rdd_time:.2f} s  ->  {rdd_time / 1:.2f} s per million rows")
print(f"DataFrame sum on 5M rows   : {df_time:.2f} s  ->  {df_time / 5:.2f} s per million rows")
print(f"DataFrame is {...:.0f}x faster per row")      # compute the ratio of the two "per million rows" figures

**✍️ Q3e** — Why is the DataFrame version so much faster per row, if both run on the same two executors? (Give at least two reasons.)

> *Write your answer here (double-click to edit, Shift+Enter to render).*

---
# Part 4 — Joins and data skew

3.2 told you: *hashing evens out keys, not popularity*. In this dataset zone 1 ("Airport") receives a large share of all trips. Let's see what that does to a **join**.

For this part we switch **AQE off** (it would try to repair some of the problems we want to observe) and use **8** shuffle partitions.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))     # the default: 10 MiB

trips = spark.read.parquet(paths["trips"])
zones = spark.read.parquet(paths["zones"])
print("zones:", zones.count(), "rows")
zones.show(3)
trips.groupBy("zone_id").count().orderBy(F.desc("count")).withColumn("share", F.round(F.col("count") / 5_000_000, 3)).show(3)

### 4a — 🔧 Broadcast hash join vs sort-merge join
Query: **revenue per district** = trips ⨝ zones, grouped by `district`. Spark can run the join in two ways: *broadcast* the small table to every executor (no shuffle of the big one), or *sort-merge* (shuffle **both** sides by the join key). The default threshold for broadcasting is `spark.sql.autoBroadcastJoinThreshold` = 10 MiB.

In [ ]:
def revenue_by_district():
    return (trips.join(zones, "zone_id")
                 .groupBy("district").agg(F.sum("fare_eur").alias("revenue")))

def join_report(label, df, reps=2):
    s0 = next_stage_id(spark)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); run(df); ts.append(time.perf_counter() - t0)
    st = new_stages(spark, s0)
    return {"strategy": label, "largest shuffle write (MB)": st.shuffle_write_MB.max(),
            "seconds (2nd run)": round(ts[-1], 2)}, st

# (1) default: let Spark choose
revenue_by_district().explain()
r1, _ = join_report("default", revenue_by_district())

# (2) forbid broadcasting -> Spark must use a sort-merge join
spark.conf.set(..., ...)                     # disable broadcast joins (what value switches the threshold off?)
revenue_by_district().explain()
r2, st_smj = join_report("sort-merge", revenue_by_district())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

pd.DataFrame([r1, r2])

**✍️ Q4a** — (i) Which join node does each plan contain, and which `Exchange` nodes appear (or disappear)? (ii) Which table was broadcast and why is Spark allowed to? (iii) Which strategy is faster here and by how much? (iv) With a `zones` table of 500 million rows, which strategy would Spark have to use?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 4b — 🔧 Seeing the skew
In the sort-merge join, all rows with the same `zone_id` go to the **same reduce task**. `stage_skew(spark, stage_id)` returns the distribution (min / quartiles / max) of the data read and the run time **per task** of a stage — the same table that the Spark UI shows on the stage page ("Summary Metrics for Completed Tasks").

In [ ]:
join_stage = int(st_smj[st_smj.shuffle_read_MB > 0].sort_values("shuffle_read_MB").stage.iloc[-1])   # the reduce side of the join
skew8 = stage_skew(spark, join_stage) # the stage of the join, with 8 partitions (the default)
print("stage", join_stage); print(skew8)
print(f"max / median shuffle read = {...:.1f}x")      # compute the ratio from the table

📸 **Screenshot 3** — open the Spark UI → *Stages* → the stage `join_stage` → **Summary Metrics for Completed Tasks** (and the task table sorted by *Shuffle Read Size*). Save it as `evidence/3_stage_skew.png`.

In [ ]:
show_evidence("evidence/3_stage_skew.png")

**Prediction.** Hot key → one overloaded task. A colleague suggests: *"just raise `spark.sql.shuffle.partitions` from 8 to 64 so the work is spread thinner"*. Write down (in Q4b below) what you expect for the **max** task. Then run the test.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "64")
r64, st64 = join_report("sort-merge, 64 partitions", revenue_by_district())
stage64 = int(st64[st64.shuffle_read_MB > 0].sort_values("shuffle_read_MB").stage.iloc[-1])
skew64 = stage_skew(spark, stage64)
print(skew64)
print(f"time: {r64['seconds (2nd run)']} s with 64 partitions   vs   {r2['seconds (2nd run)']} s with 8")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

**✍️ Q4b** — (i) What are the max/median ratios for 8 partitions? Which zone is responsible and what share of the trips does it hold? (ii) What happened to the **max** task when you used 64 partitions, and to the total time? Why? (iii) Explain, why partitioning by `hash(zone_id) mod P` cannot spread a single hot key.

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 4c — 🔧 Fixing the skew
**Fix 1 — avoid the shuffle of the big table:** tell Spark to broadcast `zones` with an explicit hint, `F.broadcast(zones)`. Measure again (time and `stage_skew` of the heaviest stage).

In [ ]:
def revenue_broadcast():
    return (trips.join(..., "zone_id")           # wrap the small table with the broadcast hint
                 .groupBy("district").agg(F.sum("fare_eur").alias("revenue")))

rb, st_b = join_report("broadcast hint", revenue_broadcast())
pd.DataFrame([r2, r64, rb])

**✍️ Q4c** — Why does the broadcast join **eliminate** the skew instead of just reducing it? When would it **not** be an option?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

**★ Bonus — Fix 2: salting.** If both tables are big you can split the hot key artificially: add a random *salt* `0…K-1` to every trip, replicate every zone row K times (once per salt), and join on `(zone_id, salt)`. Complete the code, check that the result is identical, and compare `stage_skew` and time with the plain sort-merge join.

In [ ]:
K = 8
trips_s = trips.withColumn("salt", ...)                      # random integer in 0..K-1 (hint: F.rand())
zones_s = zones.crossJoin(...)                               # every zone row repeated K times, once per salt value

def revenue_salted():
    return (trips_s.join(zones_s, [..., ...])
                   .groupBy("district").agg(F.sum("fare_eur").alias("revenue")))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
rs, st_s = join_report("salted sort-merge (K=8)", revenue_salted())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

a = revenue_salted().orderBy("district").toPandas()
b = revenue_by_district().orderBy("district").toPandas()
print("same result as the plain join:", bool(((a.revenue - b.revenue).abs() < 1e-3).all()))
stage_s = int(st_s[st_s.shuffle_read_MB > 0].sort_values("shuffle_read_MB").stage.iloc[-1])
print(stage_skew(spark, stage_s))
pd.DataFrame([r2, rs])

**✍️ ★ Q4d** — Did salting reduce the skew? Did it reduce the **total time** on your 2-core cluster? Why / why not? On what kind of cluster and data would it pay off?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

---
# Part 5 — Failures and the cost of distribution

Two questions that decide whether you should use a cluster at all:
1. **What happens when a machine dies?** (the whole point of the fault-tolerance machinery of MapReduce and Spark)
2. **What does distribution cost?** (Block 2 taught you that everything has a price)

### 5a — 🔧 A worker crashes during the reduce phase
The job below is a classic MapReduce: 8,000 records, each needing 2 ms of "real work" in the map, 16 map tasks, a shuffle into 8 partitions, and 250 ms of post-processing per key in the reduce side. (We use RDDs and `sleep` to make stages long enough to interfere with.)

First run it **twice without failures** to get the baseline.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")      # back to defaults for the rest of the lab
spark.conf.set("spark.sql.shuffle.partitions", "200")

def slow_map(x):
    time.sleep(0.002)                  # pretend each record needs 2 ms of real work
    return (x % 40, x)

def slow_post(kv):
    time.sleep(0.25)                   # ... and each result 250 ms of post-processing
    return kv

def build_job():
    return (sc.parallelize(range(8_000), 16)         # 16 input partitions -> 16 map tasks
              .map(slow_map)
              .reduceByKey(lambda a, b: a + b, 8)    # shuffle into 8 reduce partitions
              .map(slow_post))

for i in (1, 2):
    with timed(f"baseline, run {i}"):
        n = build_job().count()
print("keys:", n)

Now **arm the saboteur**: `crash_during_reduce(spark, worker=1, delay=1.5)` waits until the map stage has finished and the reduce stage is running, then — 1.5 s later — kills worker 1 *and wipes its disk* (`kill_worker(1, "all")`: the whole machine is gone, with the shuffle files that its executor had stored). Run the same job and observe.

In [ ]:
mark = next_stage_id(spark)
saboteur = ...                       # arm the saboteur BEFORE starting the job (worker 1, delay 1.5 s)
with timed("job with a worker crash"):
    n = build_job().count()
print("result:", n, "keys (must be 40)")
new_stages(spark, mark)

In [ ]:
display(executors_df(spark))
cluster_status()

**✍️ Q5a** — Read the stage table (`attempt` column!) and the executors table. (i) Was the result still correct? How much longer did the job take? (ii) Which stages have an `attempt` 1, and how many tasks does each re-attempt contain? (iii) The map stage had **already finished** when the machine died — why did Spark run part of it again? Why were the reduce tasks that had already finished **not** re-run? (iv) Which mechanism of lecture 3.2 would have avoided re-running the map tasks, and what would it cost?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

**Bonus observation** — bring the machine back with `restart_worker(1)`, wait a few seconds, and look at the executors again.

In [ ]:
restart_worker(1)
time.sleep(8)
display(cluster_status())
n = build_job().count()                  # the running application should use the new worker
executors_df(spark)

**✍️ Q5a-bis** — Did your application get a new executor? **Who** decided that — you, the driver, the master or the worker?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 5b — 🔧 The master crashes
The master is the control plane. What does it control? We crash it **3 seconds into a job**.

In [ ]:
timer = ...                                   # crash the master 3 seconds from now (hint: kill_master_after)
with timed("job while the master dies"):
    n = build_job().count()
print("result:", n, "keys")
try:
    cluster_status()
except RuntimeError as e:
    print("cluster_status() ->", e)
display(executors_df(spark)[["id", "alive", "completed_tasks"]])

Now try the two things that **do** need the master: running **another job** in the same application (this one should work — why?) and starting a **second application**. The second attempt takes about a minute to give up.

In [ ]:
with timed("another job in the SAME application"):
    print(build_job().count(), "keys")
try_new_application()

**✍️ Q5b** — (i) What survived the master's death and what did not? (ii) Why can the running application keep working? (iii) Which operations need the master? (iv) In lecture 3.4 you will study high availability: what would you add to remove this single point of failure, and is the **driver** also a single point of failure? (what would happen if this notebook's kernel died now?)

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 5c — 🔧 What does distribution cost?
Everything so far ran on a *cluster* — of two cores. Time to ask whether that was a good idea. The same aggregation (*trips longer than 1 km: count and average fare per zone*) on two single-machine tools and on your Spark cluster, for 1 M, 5 M and 20 M rows. The master died, so first restart the cluster **and measure how long that takes**.

In [ ]:
spark.stop()
t0 = time.perf_counter(); start_cluster(workers=2, cores=1, memory="1g"); cluster_restart_s = time.perf_counter() - t0
t0 = time.perf_counter(); spark = get_spark("lab33-cost"); session_restart_s = time.perf_counter() - t0
sc = spark.sparkContext
print(f"restart in this kernel (JVM already warm): cluster {cluster_restart_s:.1f} s + session {session_restart_s:.1f} s")
print(f"first start from a fresh kernel (Part 0):  cluster {cluster_start_s:.1f} s + session {session_start_s:.1f} s")

In [ ]:
def fastest(f, reps=3):
    """Best of `reps` runs, in seconds."""
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); f(); ts.append(time.perf_counter() - t0)
    return min(ts)

def cost_row(n_rows):
    p = make_dataset(f"data_{n_rows // 1_000_000}M", n_rows=n_rows, files=8)
    sql = (f"SELECT zone_id, count(*) AS n, avg(fare_eur) AS avg_fare "
           f"FROM read_parquet('{p['trips']}/*.parquet') WHERE distance_km > 1 GROUP BY zone_id")

    # --- DuckDB: a single-node, multi-threaded, columnar engine
    t_duck = fastest(lambda: duckdb.sql(sql).df())

    # --- pandas: read the needed columns, then aggregate in memory
    def pandas_query():
        df = pd.read_parquet(p["trips"], columns=["zone_id", "distance_km", "fare_eur"])
        return ...                    # keep distance_km > 1, group by zone_id, count rows (n) and mean of fare_eur (avg_fare)
    t_pandas = fastest(pandas_query)

    # --- Spark on the cluster: the 1st run on new data, then the best of 3
    qs = ...                          # the same query with the DataFrame API (see Part 3b)
    t0 = time.perf_counter(); run(qs); t_first = time.perf_counter() - t0
    t_spark = fastest(lambda: run(qs))
    return {"rows (M)": n_rows / 1e6, "DuckDB (s)": round(t_duck, 3), "pandas (s)": round(t_pandas, 3),
            "Spark (s)": round(t_spark, 2), "Spark 1st run (s)": round(t_first, 2)}

cost = pd.DataFrame([cost_row(n) for n in (1_000_000, 5_000_000, 20_000_000)])
cost["Spark / pandas"] = (cost["Spark (s)"] / cost["pandas (s)"]).round(1)
cost["Spark / DuckDB"] = (cost["Spark (s)"] / cost["DuckDB (s)"]).round(0)
cost

In [ ]:
sample = pd.read_parquet(make_dataset("data_1M", n_rows=1_000_000, files=8)["trips"])      # the whole table, 8 columns
bytes_per_row = sample.memory_usage(deep=True).sum() / len(sample)
print(f"pandas needs ~{bytes_per_row:.0f} bytes of RAM per row for the full table -> {bytes_per_row * 100e6 / 1e9:.1f} GB for 100M rows "
      f"(this Codespace has 8 GB, and a group-by makes temporary copies)")
first_s = cluster_start_s + session_start_s + cost.loc[0, "Spark 1st run (s)"]
print(f"From nothing to the first Spark answer: ~{first_s:.0f} s "
      f"(cluster {cluster_start_s:.0f} + session {session_start_s:.0f} + first query {cost.loc[0, 'Spark 1st run (s)']:.1f});   pandas or DuckDB: 0 s")
del sample

**✍️ Q5c** — (i) Is Spark ever faster than pandas in your table? Than DuckDB? (ii) What is Spark's fixed overhead per query (look at the 1 M row line), and what does that tell you about "small data"? (iii) Including the cold start of the cluster, how long do you wait before Spark answers your *first* question? Compare with a serverless query engine (3.7 preview). (iv) The real limit of pandas is not speed but ______; at what data size does a cluster become *necessary*? (v) What would change with 100 cores instead of 2?

> *Write your answer here (double-click to edit, Shift+Enter to render).*

### 5d — ✍️ Reflection (8–10 lines)
Write a short text (**8–10 lines**) answering: *"When should you NOT use Spark?"* Requirements:
- use **at least two of your own measurements** from this lab (e.g. 3d, 4a, 4b, 5c), quoting the numbers;
- say what you would use instead for a 5 GB dataset, and what would make you change your mind.
- you can cite external sources

**✍️ Q5d** — Your reflection (8–10 lines).

> *Write your answer here (double-click to edit, Shift+Enter to render).*

---
## Clean-up and submission

Stop the cluster and **stop your Codespace** (*Command Palette → "Codespaces: Stop Current Codespace"*) — a running Codespace keeps consuming your monthly quota.

In [ ]:
spark.stop()
stop_cluster()

### ✅ Submission checklist
1. The notebook **runs from top to bottom without errors** (*Restart kernel and run all*; the Part 5b cell kills the master on purpose, that is expected).
2. All 🔧 cells completed, all ✍️ cells answered **in your own words**.
3. The 3 screenshots are displayed in the notebook (`evidence/1_master_ui.png`, `2_spark_ui_plan.png`, `3_stage_skew.png`).
4. Your name, group and date are filled in at the top.
5. Submit **two files** through the virtual campus: `lab_3_3.ipynb` and its **HTML export** — in VS Code use the three-dots menu of the notebook toolbar → *Export* → *HTML*, or run `python -m jupyter nbconvert --to html lab_3_3.ipynb` in the terminal. Check that the HTML shows your screenshots and outputs. **Deadline: Wednesday 28 October 2026, 23:59** (check the virtual campus for the final date).

### Grading (10 points)
| Criterion | Points |
|---|---|
| Part 0 — cluster running, screenshot, architecture table (Q0) | 0.5 |
| Part 1 — MapReduce by hand: 1a–1d code correct, Q1.0–Q1d | 2.0 |
| Part 2 — RDDs: 2a–2c code correct, Q2a–Q2c | 1.5 |
| Part 3 — DataFrames, plans, UI, partitions: 3a–3e, Q3a–Q3e | 2.0 |
| Part 4 — Joins and skew: 4a–4c, Q4a–Q4c (★ salting is bonus, up to +0.5, total capped at 10) | 1.5 |
| Part 5 — Failure experiments, COST, reflection: 5a–5d, Q5a–Q5d | 2.0 |
| Submission compliance (runs top to bottom, `.ipynb` + `.html`, name filled in, screenshots shown) | 0.5 |

*References:* Dean & Ghemawat (2004); Zaharia et al. (2012); Armbrust et al. (2015); McSherry, Isard & Murray (2015); Rowstron et al. (2012); Lin & Dyer (2010).